<a href="https://colab.research.google.com/github/Somtochukwu438/quizgen-llama3/blob/main/02_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install transformers peft bitsandbytes accelerate datasets
from google.colab import userdata, drive
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, sys
REPO = "/content/quizgen-llama3"
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/Somtochukwu438/quizgen-llama3.git {REPO}
sys.path.append(REPO)
from src.prompts import build_messages, sciq_to_messages

OUT = "/content/drive/MyDrive/quizgen"
CKPT = f"{OUT}/checkpoints"
os.makedirs(CKPT, exist_ok=True)
print("Code loaded")

Cloning into '/content/quizgen-llama3'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 33 (delta 11), reused 7 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 44.60 KiB | 748.00 KiB/s, done.
Resolving deltas: 100% (11/11), done.
Code loaded


In [4]:
from datasets import load_dataset
sciq = load_dataset("allenai/sciq").filter(lambda x: x["support"].strip() != "")
train_raw = sciq["train"].shuffle(seed=42).select(range(2000))
val_raw = sciq["validation"].shuffle(seed=42).select(range(100))
print(len(train_raw), "train |", len(val_raw), "validation")

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

... (4 outputs left out)

Filter:   0%|          | 0/11679 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

2000 train | 100 validation


In [5]:
from transformers import AutoTokenizer
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
MAX_LEN = 1024

def tokenize(ex):
    msgs = sciq_to_messages(ex)
    prompt_text = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tok.apply_chat_template(msgs, tokenize=False)
    prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tok(full_text, add_special_tokens=False)["input_ids"]
    ok = full_ids[:len(prompt_ids)] == prompt_ids and len(full_ids) <= MAX_LEN
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids),
            "labels": labels, "ok": ok}

cols = train_raw.column_names
train_ds = train_raw.map(tokenize, remove_columns=cols).filter(lambda x: x["ok"]).remove_columns("ok")
val_ds = val_raw.map(tokenize, remove_columns=cols).filter(lambda x: x["ok"]).remove_columns("ok")
print(len(train_ds), "train |", len(val_ds), "validation after filtering")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

2000 train | 100 validation after filtering


In [6]:
ex = train_ds[0]
answer_ids = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("Total tokens:", len(ex["input_ids"]))
print("Tokens the model learns from:", len(answer_ids))
print("Learned text:", repr(tok.decode(answer_ids)))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Total tokens: 289
Tokens the model learns from: 56
Learned text: '{"question": "What do we call the system of classification consisting of a hierarchy of groupings called taxa?", "answer": "linnaean system", "distractors": ["lamarckian system", "linchpin system", "crocodilian system"]}<|eot_id|>'


In [7]:
print(repr(tok.decode(answer_ids[-10:])))

'pin system", "crocodilian system"]}<|eot_id|>'


In [8]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [11]:
import torch
from transformers import Trainer, TrainingArguments

def collate(batch):
    m = max(len(x["input_ids"]) for x in batch)
    pad = tok.pad_token_id
    return {
        "input_ids": torch.tensor([x["input_ids"] + [pad] * (m - len(x["input_ids"])) for x in batch]),
        "attention_mask": torch.tensor([x["attention_mask"] + [0] * (m - len(x["attention_mask"])) for x in batch]),
        "labels": torch.tensor([x["labels"] + [-100] * (m - len(x["labels"])) for x in batch]),
    }

args = TrainingArguments(
    output_dir=CKPT,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collate)
print("Trainer ready")

Trainer ready


In [12]:
import glob
resume = bool(glob.glob(f"{CKPT}/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume or None)

Step,Training Loss,Validation Loss
25,0.855042,0.774783
50,0.795949,0.749348
75,0.769108,0.734376
100,0.734455,0.725570
125,0.791802,0.724046


TrainOutput(global_step=125, training_loss=0.7990001125335693, metrics={'train_runtime': 844.6502, 'train_samples_per_second': 2.368, 'train_steps_per_second': 0.148, 'total_flos': 1.2849522360950784e+16, 'train_loss': 0.7990001125335693, 'epoch': 1.0})

In [13]:
import json
ADAPTER = f"{OUT}/lora_adapter"
trainer.model.save_pretrained(ADAPTER)
tok.save_pretrained(ADAPTER)
with open(f"{OUT}/train_log.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
!du -sh {ADAPTER}

110M	/content/drive/MyDrive/quizgen/lora_adapter


In [14]:
import torch
from safetensors.torch import load_file, save_file

FP16 = f"{OUT}/lora_adapter_fp16"
os.makedirs(FP16, exist_ok=True)
!cp {ADAPTER}/adapter_config.json {FP16}/
sd = load_file(f"{ADAPTER}/adapter_model.safetensors")
save_file({k: v.to(torch.float16) for k, v in sd.items()},
          f"{FP16}/adapter_model.safetensors", metadata={"format": "pt"})
!ls -lh {FP16}

total 47M
-rw------- 1 root root 1.2K Nov  1 21:34 adapter_config.json
-rw------- 1 root root  47M Nov  1 21:34 adapter_model.safetensors
